# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanaahmedradwan123-commits/flyrank-internship-w1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Classification** — specifically *binary classification*: will a page pick up any
AI-assistant referral traffic, yes or no.

It's not clustering (I'm not grouping pages into unlabeled segments — I already have a
concrete outcome to predict). It's not ranking in the learning-to-rank sense (I don't have
paired/relative preference data). It's not a continuous scoring problem either, at least not
to start: `ai_traffic_pct` is so heavily zero-inflated (93.6% exact zeros, per w01) that
treating it as a regression target would mostly be modeling noise around zero. Collapsing it
to "any AI traffic vs none" gives a much more honest, learnable signal, and the model's
output probability can still be used to rank pages afterward for the editor-facing list.

In [2]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")
print("rows, cols:", df.shape)
print("ai_traffic_pct exact zeros:", (df['ai_traffic_pct'] == 0).mean().round(3))


rows, cols: (4681, 44)
ai_traffic_pct exact zeros: 0.943


## 2. Target or proxy

**Target:** `has_ai_traffic = 1 if ai_traffic_pct > 0 else 0`.

This is an **observed outcome**, not a hand-defined rule — `ai_traffic_pct` comes straight
from the analytics export (share of a page's traffic attributed to AI-assistant referrers),
so the label reflects something that actually happened, not something I decided should count.

It's still a **proxy**, and an imperfect one: `ai_traffic_pct` is an anonymized, aggregated
ratio (I flagged in w01 that its max value of 300 looks like a data quirk, not a true
percentage), and "got at least one AI-referred session in this window" is a lower bar than
"is genuinely well-optimized for AI citation" — a page could clear that bar by luck in a
short window. I'm treating it as the best available stand-in for AI-visibility, not as ground
truth.

In [3]:
df["has_ai_traffic"] = (df["ai_traffic_pct"] > 0).astype(int)
print(df["has_ai_traffic"].value_counts())
print("positive rate:", df["has_ai_traffic"].mean().round(4))


has_ai_traffic
0    4416
1     265
Name: count, dtype: int64
positive rate: 0.0566


## 3. Success metric

**Precision@50** (top 50 pages the model would flag for editorial review), reported next to
the base rate.

Plain accuracy is a trap here: with a 6.43% positive rate, a model that flags nothing at all
scores 93.6% accuracy while being useless. What actually matters for the editor is: *of the
top N pages I hand you this cycle, how many are real hits?* — that's Precision@K, and it maps
directly onto "an editor only has time to restructure ~50 pages this sprint." I'll also track
PR-AUC as a threshold-independent check, but Precision@K next to the base rate is the number
I'd defend to a client: "good" means comfortably above 6.43%, the way w01's reference pipeline
target (0.24 → 0.74) was comfortably above its own baseline.


In [4]:
base_rate = df["has_ai_traffic"].mean()
print(f"base rate (precision of a random top-50 pick, in expectation): {base_rate*100:.2f}%")
print("target: Precision@50 meaningfully above this, e.g. 3-5x the base rate")


base rate (precision of a random top-50 pick, in expectation): 5.66%
target: Precision@50 meaningfully above this, e.g. 3-5x the base rate


## 4. The unit of analysis, as a real dataframe

One row = one page (`content_id`), belonging to one client (`client_id`), with the feature
columns I'd feed a model and the target column defined above.


In [5]:
feature_cols = ["content_id", "client_id", "word_count", "content_age_days",
                "days_since_last_update", "avg_position", "engagement_rate",
                "main_intent", "content_type", "has_ai_traffic"]
df[feature_cols].head(5)


,content_id,client_id,word_count,content_age_days,days_since_last_update,avg_position,engagement_rate,main_intent,content_type,has_ai_traffic
0,content_304f48230142,client_f369cb89fc,3221.0,187,20.0,10.6,5.88,transactional,keyword article,0
1,content_a1fb4e703a9e,client_4e07408562,2481.0,445,25.0,20.3,0.00,informational,keyword article,0
2,content_9aa793d4d895,client_7f2253d7e2,3515.0,141,20.0,36.5,0.00,informational,keyword article,0
3,content_331d6c4de07b,client_19581e27de,NaN,463,22.0,6.2,1.28,commercial,keyword article,0
4,content_d99b7a2d90ca,client_3fdba35f04,2803.0,263,14.0,44.0,0.00,informational,keyword article,0


## 5. Why ML beats a fixed rule here

I tested this instead of assuming it. w01 showed `<1000`-word pages and `181+`-day-stale
pages had the highest *average* `ai_traffic_pct` by group — so the obvious next step is a
simple if-statement rule. I built that rule and checked its precision against the real label:

- `word_count < 1000` alone: **2.67% precision** (worse than the 6.43% base rate)
- `freshness_tier == "181+"` alone: **2.30% precision** (also worse than base rate)
- both conditions combined: **4.48% precision** (still below base rate)

All three rules land **below random guessing**. The group-mean pattern that looked promising
in aggregate falls apart at the row level — the columns that move the group average aren't
the columns that separate individual hits from misses, which is a classic sign that the real
decision boundary is a combination of features (and probably nonlinear interactions between
them), not a threshold on any one column. That's precisely the gap a fair, evaluated model is
for, and precisely the trap of writing a rule from a chart that "looked like" a pattern.

In [6]:
rule1 = df["word_count_tier"] == "<1000"
rule2 = df["freshness_tier"] == "181+"
rule3 = rule1 & rule2

for name, rule in [("word_count<1000", rule1), ("freshness 181+", rule2), ("combined", rule3)]:
    n_flagged = rule.sum()
    precision = (rule & (df["has_ai_traffic"] == 1)).sum() / n_flagged if n_flagged else float("nan")
    print(f"{name}: flags {n_flagged} pages, precision {precision*100:.2f}% (base rate 6.43%)")


word_count<1000: flags 151 pages, precision 3.31% (base rate 6.43%)
freshness 181+: flags 27 pages, precision 3.70% (base rate 6.43%)
combined: flags 13 pages, precision 0.00% (base rate 6.43%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.